In [1]:
# ============================================================
# NETFLIX MOVIE RECOMMENDATION SYSTEM - CONTENT BASED FILTERING
# Complete Code - Based on TMDB 5000 Dataset
# ============================================================

In [2]:
#  Import Libraries and Datasets

In [3]:
import numpy as np
import pandas as pd
import ast
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [4]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [5]:
movies = movies.merge(credits, on='title')
print("After merge shape:", movies.shape)  # (4809, 23+4-1 = 26)

After merge shape: (4809, 23)


In [6]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
print(movies.shape)   # (4809, 7)
movies.head(3)

(4809, 7)


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [7]:
print(movies.isnull().sum())

# Drop rows with null values
movies.dropna(inplace=True)
print("After dropping nulls:", movies.shape)

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64
After dropping nulls: (4806, 7)


In [8]:
print("Duplicates:", movies.duplicated().sum())

Duplicates: 0


In [9]:
movies.head(2)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [10]:
print(movies['genres'][0])

[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]


In [11]:
def convert(text):
    lst = []
    for i in ast.literal_eval(text):
        lst.append(i['name'])
    return lst

# Apply to genres and keywords
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

print(movies['genres'][0])    # ['Action', 'Adventure', 'Fantasy', 'Science Fiction']
print(movies['keywords'][0])  # ['culture clash', 'future', 'space war', ...]

['Action', 'Adventure', 'Fantasy', 'Science Fiction']
['culture clash', 'future', 'space war', 'space colony', 'society', 'space travel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alien planet', 'cgi', 'marine', 'soldier', 'battle', 'love affair', 'anti war', 'power relations', 'mind and soul', '3d']


In [12]:
print(movies['cast'][0])
print("Number of cast members in Avatar:", len(ast.literal_eval(movies['cast'][0])))  # 83

[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 2

In [13]:
def fetch_cast(text):
    lst = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter != 3:
            lst.append(i['name'])
            counter += 1
        else:
            break
    return lst

# Test it
print(fetch_cast(movies['cast'][0]))  # ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']


In [14]:
movies['cast'] = movies['cast'].apply(fetch_cast)

In [15]:
print(movies['crew'][0])
print("Number of crew members in Avatar:", len(ast.literal_eval(movies['crew'][0])))  # 153

[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"}, 

In [16]:
def fetch_director(text):
    lst = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            lst.append(i['name'])
    return lst

# Test it
print(fetch_director(movies['crew'][0]))  # ['James Cameron']

['James Cameron']


In [17]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [18]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Verify
print(movies['overview'][0])  # ['In', 'the', '22nd', 'century', ...]

['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is', 'dispatched', 'to', 'the', 'moon', 'Pandora', 'on', 'a', 'unique', 'mission,', 'but', 'becomes', 'torn', 'between', 'following', 'orders', 'and', 'protecting', 'an', 'alien', 'civilization.']


In [19]:
# ============================================================
# CELL 17 - Remove spaces from multi-word names
# This is CRITICAL so that "Sam Worthington" doesn't get split
# into separate tokens "Sam" and "Worthington"
# "Sam Worthington" → "SamWorthington" (treated as one token)
# ============================================================
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

# Verify
print(movies['cast'][0])     # ['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver']
print(movies['crew'][0])     # ['JamesCameron']

['SamWorthington', 'ZoeSaldana', 'SigourneyWeaver']
['JamesCameron']


In [20]:
# ============================================================
# CELL 18 - Create the 'tags' column with FEATURE WEIGHTING
# Multiply the lists to duplicate the text, forcing the model to prioritize them
# ============================================================
movies['crew'] = movies['crew'].apply(lambda x: x * 3)        # 3x weight for Director
movies['cast'] = movies['cast'].apply(lambda x: x * 2)        # 2x weight for Cast
movies['genres'] = movies['genres'].apply(lambda x: x * 2)    # 2x weight for Genres
movies['keywords'] = movies['keywords'].apply(lambda x: x * 2)# 2x weight for Keywords

# Combine everything into one list (Overview stays at 1x weight)
movies['tags'] = (movies['overview'] +
                  movies['genres'] +
                  movies['keywords'] +
                  movies['cast'] +
                  movies['crew'])

# Preview
print(movies['tags'][0])


['In', 'the', '22nd', 'century,', 'a', 'paraplegic', 'Marine', 'is', 'dispatched', 'to', 'the', 'moon', 'Pandora', 'on', 'a', 'unique', 'mission,', 'but', 'becomes', 'torn', 'between', 'following', 'orders', 'and', 'protecting', 'an', 'alien', 'civilization.', 'Action', 'Adventure', 'Fantasy', 'ScienceFiction', 'Action', 'Adventure', 'Fantasy', 'ScienceFiction', 'cultureclash', 'future', 'spacewar', 'spacecolony', 'society', 'spacetravel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alienplanet', 'cgi', 'marine', 'soldier', 'battle', 'loveaffair', 'antiwar', 'powerrelations', 'mindandsoul', '3d', 'cultureclash', 'future', 'spacewar', 'spacecolony', 'society', 'spacetravel', 'futuristic', 'romance', 'space', 'alien', 'tribe', 'alienplanet', 'cgi', 'marine', 'soldier', 'battle', 'loveaffair', 'antiwar', 'powerrelations', 'mindandsoul', '3d', 'SamWorthington', 'ZoeSaldana', 'SigourneyWeaver', 'SamWorthington', 'ZoeSaldana', 'SigourneyWeaver', 'JamesCameron', 'JamesCameron', 'Jame

In [21]:
# ============================================================
# CELL 19 - Create final dataframe with only needed columns
# ============================================================
data = movies[['movie_id', 'title', 'tags']]
data = data.copy()  # avoid SettingWithCopyWarning

# Convert tags list to a single string (lowercase)
data['tags'] = data['tags'].apply(lambda x: " ".join(x).lower())

# Preview
print(data['tags'][0])
# "in the 22nd century, a parapleg marin is dispa..."
data.head()

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver samworthington zoesaldana sigourneyweaver jamescameron jamescameron jamescameron


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [22]:
# CELL 20 - Apply Stemming to the tags
# Stemming reduces words to their root form:
# "loved", "loving", "loves" → "love"
# "action", "actions", "acting" → "act"
# This improves matching between similar movies
# ============================================================
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

data['tags'] = data['tags'].apply(stem)

# Verify
print(data['tags'][0])
# Words are now stemmed: "paraplegic" → "parapleg"


in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav samworthington zoesaldana sigourneyweav jamescameron jamescameron jamescameron


In [23]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [24]:
#vector embedding
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load a highly efficient pre-trained semantic model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Generate embeddings using 'data' instead of 'new_df'
print("Generating semantic vector embeddings...")
vectors = embedding_model.encode(data['tags'].tolist(), show_progress_bar=True)

# 3. Calculate the new semantic cosine similarity matrix
similarity = cosine_similarity(vectors)

# 4. Save your updated matrix using your existing bz2 compression code
import bz2
import pickle
with bz2.BZ2File('similarity.pbz2', 'w') as f:
    pickle.dump(similarity, f)

Generating semantic vector embeddings...


Batches:   0%|          | 0/151 [00:00<?, ?it/s]

In [26]:
# ============================================================
# CELL 22 - Generate Semantic Embeddings and Calculate Similarity
# (This replaces the old CountVectorizer and exploration cells)
# ============================================================
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import bz2
import pickle

print("Loading AI Model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating semantic vector embeddings...")
vectors = embedding_model.encode(data['tags'].tolist(), show_progress_bar=True)

print("Calculating similarity matrix...")
similarity = cosine_similarity(vectors)

print("Saving similarity.pbz2...")
with bz2.BZ2File('similarity.pbz2', 'w') as f:
    pickle.dump(similarity, f)
    
print("Successfully saved similarity.pbz2!")

Loading AI Model...
Generating semantic vector embeddings...


Batches:   0%|          | 0/151 [00:00<?, ?it/s]

Calculating similarity matrix...
Saving similarity.pbz2...
Successfully saved similarity.pbz2!


In [27]:
# ============================================================
# CELL 23 - The Recommendation Function 
# (Formerly Cell 33)
# ============================================================
def recommend(movie):
    # Step 1: Find the index of the input movie in our dataframe
    movie_index = data[data['title'] == movie].index[0]

    # Step 2: Get similarity scores for this movie with all others
    distances = similarity[movie_index]

    # Step 3: Sort all movies by similarity score (descending)
    # [1:6] skips the first result (the movie itself with score 1.0)
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    # Step 4: Print the top 5 recommended movies
    print(f"\nMovies similar to '{movie}':")
    for i in movies_list:
        print(" -", data.iloc[i[0]].title)

In [28]:
# ============================================================
# CELL 24 - Test the Model
# ============================================================
recommend('The Conjuring')


Movies similar to 'The Conjuring':
 - Demonic
 - Halloween
 - The Conjuring 2
 - The Visit
 - 8MM


In [29]:
# Test the recommend() function

In [30]:
recommend('Avatar')


Movies similar to 'Avatar':
 - Battle: Los Angeles
 - The Inhabited Island
 - Aliens
 - Prometheus
 - The Day the Earth Stood Still


In [31]:

recommend('Batman v Superman: Dawn of Justice')


Movies similar to 'Batman v Superman: Dawn of Justice':
 - Batman Begins
 - Batman & Robin
 - The Dark Knight
 - Superman
 - Batman


In [32]:
recommend('The Dark Knight Rises')


Movies similar to 'The Dark Knight Rises':
 - Batman Forever
 - Batman Returns
 - Batman Begins
 - The Dark Knight
 - Batman


In [33]:
print(list(data['title'].values))

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies', 'The Amazing Spider-Man', 'Robin Hood', 'The Hobbit: The Desolation of Smaug', 'The Golden Compass', 'King Kong', 'Titanic', 'Captain America: Civil War', 'Battleship', 'Jurassic World', 'Skyfall', 'Spider-Man 2', 'Iron Man 3', 'Alice in Wonderland', 'X-Men: The Last Stand', 'Monsters University', 'Transformers: Revenge of the Fallen', 'Transformers: Age of Extinction', 'Oz: The Great and Powerful', 'The Amazing Spider-Man 2', 'TRON: Legacy', 'Cars 2', 'Green Lant

In [38]:
movies_dict = data.to_dict()
print(list(movies_dict.keys())[:5])

['movie_id', 'title', 'tags']


In [42]:
# ============================================================
# CELL 39 - Save the final files for Streamlit
# ============================================================
import pickle

# Convert the dataframe to a dictionary first, then save it
pickle.dump(data.to_dict(), open('movie_dict.pkl', 'wb'))

print("Model saved successfully!")
print("File created: movie_dict.pkl")

Model saved successfully!
File created: movie_dict.pkl


In [43]:
loaded_movies = pickle.load(open('movies.pkl', 'rb'))
loaded_similarity = pickle.load(open('similarity.pkl', 'rb'))

print("Movies shape:", loaded_movies.shape)
print("Similarity shape:", loaded_similarity.shape)
print("Test recommend after loading:")
recommend('Avatar')

Movies shape: (4806, 3)
Similarity shape: (4806, 4806)
Test recommend after loading:

Movies similar to 'Avatar':
 - Battle: Los Angeles
 - The Inhabited Island
 - Aliens
 - Prometheus
 - The Day the Earth Stood Still


In [44]:
# ============================================================
# Test loading the compressed files
# ============================================================
import pickle
import bz2
import pandas as pd

print("Testing file loading...")

# Load the dictionary and convert back to dataframe
loaded_dict = pickle.load(open('movie_dict.pkl', 'rb'))
loaded_movies = pd.DataFrame(loaded_dict)

# Load the compressed similarity matrix
with bz2.BZ2File('similarity.pbz2', 'rb') as f:
    loaded_similarity = pickle.load(f)

print("Movies shape:", loaded_movies.shape)
print("Similarity shape:", loaded_similarity.shape)
print("✅ Files loaded successfully!")

Testing file loading...
Movies shape: (4806, 3)
Similarity shape: (4806, 4806)
✅ Files loaded successfully!
